# GRPO Reinforcement Learning - Chatbot Tim Legal (PGABL) [Advanced / Opsional]
**Nama:** Stanley Nathanael Wijaya

Notebook ini melanjutkan model hasil `Fine-tuning_submission_PGABL_Stanley-Nathanael-Wijaya.ipynb`
dengan **GRPO (Group Relative Policy Optimization)** memakai `GRPOTrainer` dari TRL + Unsloth, agar
model belajar menampilkan proses berpikir (`<think>...</think>`) sebelum menjawab.

**PENTING:** GRPO membangkitkan banyak *completion* per prompt (`num_generations`) sehingga jauh
lebih berat dari SFT biasa. Jalankan di Colab/Kaggle dengan GPU >= T4 16GB. Parameter
`num_generations` dan `max_completion_length` sengaja dibuat kecil untuk memitigasi OOM.


## 1. Instalasi & Load Model Hasil Fine-tuning

Sama seperti notebook Fine-tuning: cukup `pip install unsloth`, tidak perlu install `trl`/`peft`/
`bitsandbytes` manual (Colab + Unsloth sudah menyelaraskan versinya).


In [1]:
%%capture
!pip install unsloth
!pip install -q datasets wandb rouge-score

In [2]:
import os
import re
from getpass import getpass

import torch
from huggingface_hub import login

SEED = 3407
torch.manual_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU tidak terdeteksi. Di Colab: Runtime > Change runtime type > pilih T4 GPU."
    )


def get_secret(name, prompt, optional=False):
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        return value
    value = getpass(prompt)
    if not value and not optional:
        raise ValueError(f"{name} wajib diisi.")
    return value


HF_TOKEN = get_secret("HF_TOKEN", "Masukkan Hugging Face Write Token: ")
login(token=HF_TOKEN)

HF_USERNAME = os.environ.get("HF_USERNAME") or input("Masukkan username Hugging Face kamu: ")
FT_REPO_ID = os.environ.get("FT_REPO_ID") or f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-id"
GRPO_REPO_ID = f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-id-grpo"
print("Memuat kembali model instruct hasil fine-tuning dari:", FT_REPO_ID)

CUDA available: True
Masukkan Hugging Face Write Token: ··········
Masukkan username Hugging Face kamu: xStyNWx
Memuat kembali model instruct hasil fine-tuning dari: xStyNWx/qwen2.5-1.5b-legal-chatbot-id


In [3]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FT_REPO_ID,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 2. Dataset & System Prompt

Menggunakan dataset yang sama (`Ichsan2895/alpaca-gpt4-indonesian`) sesuai ketentuan submission
(dataset untuk fine-tuning **dan** GRPO wajib dataset yang sama). Prompt sistem diarahkan agar model
selalu berpikir di dalam tag `<think>...</think>` sebelum menjawab.


In [4]:
from datasets import load_dataset

GRPO_SYSTEM_PROMPT = (
    "Kamu adalah asisten AI internal Tim Legal perusahaan. Sebelum menjawab, tuliskan proses "
    "berpikirmu di dalam tag <think>...</think>, lalu berikan jawaban akhir dalam Bahasa Indonesia "
    "berdasarkan konteks/pertanyaan yang diberikan."
)

raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")


def normalize_to_alpaca(example):
    return {"instruction": example["input"], "input": "", "output": example["output"]}


if "instruction" not in raw_dataset.column_names:
    raw_dataset = raw_dataset.map(normalize_to_alpaca, remove_columns=raw_dataset.column_names)

# Subset agar eksperimen GRPO tidak berjalan terlalu lama
grpo_dataset = raw_dataset.shuffle(seed=SEED).select(range(2000))


def to_grpo_prompt(example):
    prompt = [
        {"role": "system", "content": GRPO_SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
    ]
    return {
        "prompt": tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True),
        "ground_truth": example["output"],
    }


grpo_dataset = grpo_dataset.map(to_grpo_prompt)
print(grpo_dataset[0]["prompt"])

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv: reconstructing file:   0%|          |  0.00B / 41.4MB            

alpaca-gpt4-indonesia.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

<|im_start|>system
Kamu adalah asisten AI internal Tim Legal perusahaan. Sebelum menjawab, tuliskan proses berpikirmu di dalam tag <think>...</think>, lalu berikan jawaban akhir dalam Bahasa Indonesia berdasarkan konteks/pertanyaan yang diberikan.<|im_end|>
<|im_start|>user
Apa tiga prinsip terpenting yang harus dipertimbangkan saat membuat infografis?
<|im_end|>
<|im_start|>assistant



## 3. Reward Functions

Empat reward function sesuai ketentuan kriteria Advanced.


In [5]:
THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)


def _think_matches(text):
    return THINK_RE.findall(text)


def format_reward_func(completions, **kwargs):
    """Reward shaping bertahap (maks +1.0), penalti -0.5 jika tag <think>/</think> duplikat."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        score = 0.0
        opens = text.count("<think>")
        closes = text.count("</think>")

        if opens >= 1:
            score += 0.2
        if closes >= 1:
            score += 0.3

        matches = _think_matches(text)
        starts_with_think = text.strip().startswith("<think>")
        well_closed = len(matches) == 1 and "</think>" in text
        followed_by_answer = bool(re.search(r"</think>\s*\S+", text, re.DOTALL))
        if starts_with_think and well_closed and followed_by_answer:
            score = 1.0

        if opens > 1 or closes > 1:
            score -= 0.5

        rewards.append(score)
    return rewards


def reasoning_length_reward(completions, **kwargs):
    """Poin proporsional berdasarkan panjang isi <think>...</think>."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        matches = _think_matches(text)
        if not matches:
            # toleran terhadap think yang terpotong batas token (tag <think> ada, </think> belum muncul)
            if "<think>" in text:
                content = text.split("<think>", 1)[1].strip()
            else:
                rewards.append(0.0)
                continue
        else:
            content = matches[0].strip()

        if not content:
            rewards.append(0.0)
        elif len(content) < 50:
            rewards.append(0.2)
        elif len(content) < 200:
            rewards.append(0.5)
        else:
            rewards.append(1.0)
    return rewards


def _final_answer(text):
    return THINK_RE.sub("", text).strip()


def correctness_reward(completions, ground_truth, **kwargs):
    """+1.0 jika jawaban akhir mengandung/mirip ground truth (ROUGE-L)."""
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rewards = []
    for completion, truth in zip(completions, ground_truth):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        answer = _final_answer(text)
        if not answer:
            rewards.append(0.0)
            continue
        if truth.strip() and truth.strip() in answer:
            rewards.append(1.0)
            continue
        score = scorer.score(truth, answer)["rougeL"].fmeasure
        rewards.append(1.0 if score >= 0.5 else score)
    return rewards


ENGLISH_HINT_RE = re.compile(r"\b(the|is|are|and|of|to|this|that|with)\b", re.IGNORECASE)


def language_reward_func(completions, **kwargs):
    """-0.5 jika terdeteksi Bahasa Inggris, +1.0 jika murni Bahasa Indonesia."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        answer = _final_answer(text)
        english_hits = len(ENGLISH_HINT_RE.findall(answer))
        rewards.append(-0.5 if english_hits >= 3 else 1.0)
    return rewards

## 4. GRPOTrainer

In [7]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="outputs/grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,          # kecil supaya menghindari OOM di GPU terbatas
    max_completion_length=256,  # dibatasi untuk mengontrol VRAM
    max_prompt_length=512,
    max_steps=200,
    logging_steps=5,
    save_steps=100,
    seed=SEED,
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
    args=grpo_config,
    train_dataset=grpo_dataset,
)
grpo_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/pyt

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
5,0.000000,1.265178,0.100546,170.700000,106.600000,214.000000,0.350000,106.233334,55.400000,149.800000,0.000017,0.000000,0.000000,0.000000,0.000000,0.265178,0.100546,1.000000,0.000000
10,0.000001,1.283703,0.233329,175.400000,96.200000,221.000000,0.450000,126.800000,96.200000,156.200000,0.000017,0.050000,0.100000,0.025000,0.050000,0.208703,0.097612,1.000000,0.000000
15,-0.000000,1.188446,0.046003,151.500000,104.000000,205.600000,0.350000,85.883334,52.800000,135.000000,0.000019,0.000000,0.000000,0.000000,0.000000,0.188446,0.046003,1.000000,0.000000
20,-0.000000,1.156609,0.045598,122.300000,73.000000,189.000000,0.100000,111.583337,73.000000,154.600000,0.000024,0.000000,0.000000,0.000000,0.000000,0.156609,0.045598,1.000000,0.000000
25,0.000001,1.387355,0.211430,185.400000,142.200000,253.800000,0.400000,112.650002,91.000000,143.200000,0.000028,0.125000,0.095743,0.125000,0.095743,0.137355,0.056648,1.000000,0.000000
30,0.000000,1.131645,0.022629,195.850000,170.800000,223.400000,0.550000,83.766669,68.400000,104.400000,0.000024,0.000000,0.000000,0.000000,0.000000,0.131645,0.022629,1.000000,0.000000
35,0.000002,1.147835,0.061068,123.950000,53.800000,184.800000,0.150000,103.066669,53.800000,155.400000,0.000049,0.000000,0.000000,0.000000,0.000000,0.147835,0.061068,1.000000,0.000000
40,-0.000000,1.234777,0.114280,188.100000,94.200000,225.600000,0.550000,116.250000,94.200000,132.600000,0.000025,0.000000,0.000000,0.000000,0.000000,0.234777,0.114280,1.000000,0.000000
45,-0.000001,1.161318,0.037400,175.250000,119.400000,211.000000,0.500000,61.450000,17.000000,85.600000,0.000083,0.000000,0.000000,0.000000,0.000000,0.161318,0.037400,1.000000,0.000000
50,0.000000,1.230028,0.047168,159.750000,135.600000,183.200000,0.350000,102.700000,84.400000,124.200000,0.000044,0.000000,0.000000,0.000000,0.000000,0.230028,0.047168,1.000000,0.000000


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
5,0.000000,1.265178,0.100546,170.700000,106.600000,214.000000,0.350000,106.233334,55.400000,149.800000,0.000017,0.000000,0.000000,0.000000,0.000000,0.265178,0.100546,1.000000,0.000000
10,0.000001,1.283703,0.233329,175.400000,96.200000,221.000000,0.450000,126.800000,96.200000,156.200000,0.000017,0.050000,0.100000,0.025000,0.050000,0.208703,0.097612,1.000000,0.000000
15,-0.000000,1.188446,0.046003,151.500000,104.000000,205.600000,0.350000,85.883334,52.800000,135.000000,0.000019,0.000000,0.000000,0.000000,0.000000,0.188446,0.046003,1.000000,0.000000
20,-0.000000,1.156609,0.045598,122.300000,73.000000,189.000000,0.100000,111.583337,73.000000,154.600000,0.000024,0.000000,0.000000,0.000000,0.000000,0.156609,0.045598,1.000000,0.000000
25,0.000001,1.387355,0.211430,185.400000,142.200000,253.800000,0.400000,112.650002,91.000000,143.200000,0.000028,0.125000,0.095743,0.125000,0.095743,0.137355,0.056648,1.000000,0.000000
30,0.000000,1.131645,0.022629,195.850000,170.800000,223.400000,0.550000,83.766669,68.400000,104.400000,0.000024,0.000000,0.000000,0.000000,0.000000,0.131645,0.022629,1.000000,0.000000
35,0.000002,1.147835,0.061068,123.950000,53.800000,184.800000,0.150000,103.066669,53.800000,155.400000,0.000049,0.000000,0.000000,0.000000,0.000000,0.147835,0.061068,1.000000,0.000000
40,-0.000000,1.234777,0.114280,188.100000,94.200000,225.600000,0.550000,116.250000,94.200000,132.600000,0.000025,0.000000,0.000000,0.000000,0.000000,0.234777,0.114280,1.000000,0.000000
45,-0.000001,1.161318,0.037400,175.250000,119.400000,211.000000,0.500000,61.450000,17.000000,85.600000,0.000083,0.000000,0.000000,0.000000,0.000000,0.161318,0.037400,1.000000,0.000000
50,0.000000,1.230028,0.047168,159.750000,135.600000,183.200000,0.350000,102.700000,84.400000,124.200000,0.000044,0.000000,0.000000,0.000000,0.000000,0.230028,0.047168,1.000000,0.000000


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

TrainOutput(global_step=200, training_loss=2.737436443567276e-07, metrics={'train_runtime': 4025.5434, 'train_samples_per_second': 0.199, 'train_steps_per_second': 0.05, 'total_flos': 0.0, 'train_loss': 2.737436443567276e-07})

## 5. Push Model GRPO ke Hugging Face Hub

In [8]:
model.push_to_hub_merged(
    GRPO_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)
print("Model GRPO berhasil di-push ke:", f"https://huggingface.co/{GRPO_REPO_ID}")

with open("link_huggingface.txt", "a") as f:
    f.write(f"GRPO model: https://huggingface.co/{GRPO_REPO_ID}\n")

Unsloth: Restored added_tokens_decoder metadata in xStyNWx/qwen2.5-1.5b-legal-chatbot-id-grpo/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `xStyNWx/qwen2.5-1.5b-legal-chatbot-id-grpo`: 100%|██████████| 1/1 [00:52<00:00, 52.26s/it]


Successfully copied all 1 files from cache to `xStyNWx/qwen2.5-1.5b-legal-chatbot-id-grpo`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...id-grpo/model.safetensors:   1%|          | 15.9MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:38<00:00, 98.46s/it]


Unsloth: Merge process complete. Saved to `/content/xStyNWx/qwen2.5-1.5b-legal-chatbot-id-grpo`
Model GRPO berhasil di-push ke: https://huggingface.co/xStyNWx/qwen2.5-1.5b-legal-chatbot-id-grpo


## 6. Test Case Wajib

Prompt: *"Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang
lembur?"* — output harus menampilkan proses reasoning di dalam tag `<think>`.


In [11]:
FastLanguageModel.for_inference(model)

test_prompt = [
    {"role": "system", "content": GRPO_SYSTEM_PROMPT},
    {"role": "user", "content": (
        "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. "
        "Apakah saya berhak dapat uang lembur?"
    )},
]
inputs = tokenizer.apply_chat_template(
    test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

outputs = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
result = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(result)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sebagai atasan, jika Anda telah mengumpulkan semua bukti dan memastikan bahwa itu benar-benar merupakan keputusan dari tim Anda atau organisasi, maka Anda memiliki hak untuk mendapatkan upah tambahan. Namun, sebaiknya pertimbangkan dengan bijak apakah ini akan menjadi pengalaman yang membosankan bagi Anda, karena selama periode waktu tertentu, Anda mungkin tidak ingin bekerja lebih banyak waktu. Selain itu, pastikan bahwa Anda sudah menulis catatan tentang apa yang terjadi dan bagaimana itu berjalan sehingga Anda bisa memberitahu manajer Anda ketika mereka melihatnya.
